# Flow vs. Initial Structure Visualization
Run the cells below to load a chosen `sid` from the LMDB and the corresponding Flow output, then render both structures inline via `nglview`.

In [ ]:
# Ensure interactive visualization dependencies are installed
import sys, subprocess

import nglview
from IPython.display import display

In [ ]:
# Configure paths and load structures
from pathlib import Path
import ase.io
import numpy as np
from ase import Atoms
from adsorbdiff.datasets.lmdb_dataset import LmdbDataset

sid = "68_700_14"  # change to the system you want to inspect
lmdb_path = Path("val_nonrelaxed_update")
flow_dir = Path("grid_search_runs/cfg1_steps100/0")
flow_traj_path = flow_dir / f"{sid}.traj"
initial_traj_path = flow_dir / f"{sid}_initial.traj"  # optional pre-exported initial frame

if not flow_traj_path.exists():
    raise FileNotFoundError(f"Flow trajectory not found: {flow_traj_path}")

def load_initial_from_lmdb(root: Path, target_sid: str) -> Atoms:
    dataset = LmdbDataset({"src": str(root), "key_mapping": {"y": "energy", "force": "forces"}})
    for record in dataset:
        if str(record.sid) != target_sid:
            continue
        atoms = Atoms(numbers=record.atomic_numbers.numpy())
        atoms.set_positions(record.pos.numpy())
        atoms.set_tags(record.tags.numpy())
        cell = record.cell
        if hasattr(cell, "numpy"):
            cell = cell.numpy()
        cell = np.asarray(cell)
        if cell.shape == (3, 3):
            pass
        elif cell.shape == (3,):
            cell = np.diag(cell)
        elif cell.size == 9:
            cell = cell.reshape(3, 3)
        else:
            raise ValueError(f"Unexpected cell shape for {target_sid}: {cell.shape}")
        atoms.set_cell(cell)
        atoms.set_pbc([True, True, True])
        return atoms
    raise KeyError(f"SID {target_sid} not found in {root}")

try:
    initial_atoms = ase.io.read(initial_traj_path.as_posix())
    initial_source = f"file: {initial_traj_path}"
except (FileNotFoundError, OSError):
    initial_atoms = load_initial_from_lmdb(lmdb_path, sid)
    initial_source = f"lmdb: {lmdb_path}"

flow_atoms = ase.io.read(flow_traj_path.as_posix(), index=-1)
print(f"Loaded initial atoms from {initial_source}")
print(f"Loaded flow trajectory: {flow_traj_path}")

In [ ]:
# Display initial and Flow structures inline
import nglview as nv

view_initial = nv.show_ase(initial_atoms)
view_flow = nv.show_ase(flow_atoms)

print(f"Initial structure ({initial_source})")
display(view_initial)

print("Flow structure")
display(view_flow)